In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from flaml import AutoML
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import PartialDependenceDisplay
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os
from pathlib import Path

In [ ]:
BASE = Path().resolve()
if BASE.name == 'notebooks':
    BASE = BASE.parent

In [ ]:
os.makedirs(BASE / 'outputs', exist_ok=True)

In [ ]:
df = pd.read_csv(BASE / 'data/processed/clean_dataset.csv')

In [ ]:
FEATURES = ['grid_position', 'Q1_s', 'Q2_s', 'Q3_s', 'delta_to_pole_s', 'q_session_reached']
TARGET = 'finish_position'

In [ ]:
# time based split, train on 2017-2023 test on 2024-2025
train = df[df['season'] <= 2023]
test  = df[df['season'] >= 2024]

In [ ]:
X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

In [ ]:
# baseline is just predicting grid position as finish position
baseline_preds = X_test['grid_position'].values

In [ ]:
# scale, fit on train only
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES)

In [ ]:
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
baseline_mae  = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = mean_squared_error(y_test, baseline_preds) ** 0.5
baseline_r, _ = stats.spearmanr(y_test, baseline_preds)
print(f'Baseline MAE: {baseline_mae:.3f}  RMSE: {baseline_rmse:.3f}  rho: {baseline_r:.3f}')

In [ ]:
# AutoML, let flaml search for 120 seconds
automl = AutoML()
automl.fit(X_train_scaled, y_train, task='regression', metric='mae',
           time_budget=120, seed=42, verbose=0)

In [ ]:
print(f'Best estimator: {automl.best_estimator}')
automl_preds = automl.predict(X_test_scaled)
automl_r, _  = stats.spearmanr(y_test, automl_preds)
print(f'AutoML MAE: {mean_absolute_error(y_test, automl_preds):.3f}  rho: {automl_r:.3f}')

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(),
    'Lasso':             Lasso(),
    'Decision Tree':     DecisionTreeRegressor(random_state=42),
    'KNN':               KNeighborsRegressor(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Voting Ensemble':   VotingRegressor(estimators=[
        ('lr', LinearRegression()),
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ]),
    'Stacking Ensemble': StackingRegressor(
        estimators=[
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
        ],
        final_estimator=Ridge(), cv=5,
    ),
}

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    r, _  = stats.spearmanr(y_test, preds)
    results[name] = {
        'MAE':          round(mean_absolute_error(y_test, preds), 3),
        'RMSE':         round(mean_squared_error(y_test, preds) ** 0.5, 3),
        'Spearman rho': round(r, 3)
    }

In [ ]:
results[f'AutoML ({automl.best_estimator})'] = {
    'MAE':          round(mean_absolute_error(y_test, automl_preds), 3),
    'RMSE':         round(mean_squared_error(y_test, automl_preds) ** 0.5, 3),
    'Spearman rho': round(automl_r, 3)
}

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Spearman rho', ascending=False)
print(f'\nBaseline MAE: {baseline_mae:.3f}  rho: {baseline_r:.3f}')
print(results_df.to_string())
results_df.to_csv(BASE / 'outputs/model_results.csv')

In [ ]:
# tune gradient boosting, using TimeSeriesSplit so folds respect time order
param_grid = {'n_estimators':  [100, 200, 300],
              'learning_rate': [0.05, 0.1, 0.2],
              'max_depth':     [2, 3, 4]}

In [ ]:
tscv   = TimeSeriesSplit(n_splits=5)
search = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid,
                      scoring='neg_mean_absolute_error', cv=tscv, n_jobs=-1)
search.fit(X_train_scaled, y_train)
tuned_model = search.best_estimator_

In [ ]:
tuned_preds = tuned_model.predict(X_test_scaled)
tuned_r, _  = stats.spearmanr(y_test, tuned_preds)
print(f'Best params: {search.best_params_}')
print(f'Tuned GB MAE: {mean_absolute_error(y_test, tuned_preds):.3f}  rho: {tuned_r:.3f}')

In [ ]:
# pick whichever GB version had the better rho
default_gb    = models['Gradient Boosting']
default_preds = default_gb.predict(X_test_scaled)
default_r, _  = stats.spearmanr(y_test, default_preds)
best_gb_model = tuned_model if tuned_r >= default_r else default_gb
best_gb_preds = tuned_preds if tuned_r >= default_r else default_preds
best_gb_r     = max(tuned_r, default_r)
best_gb_label = 'Tuned GB' if tuned_r >= default_r else 'Default GB (tuning did not improve rho)'
print(f'Best GB: {best_gb_label}  rho: {best_gb_r:.3f}')

In [ ]:
# model comparision bar chart
plot_df = results_df.reset_index().rename(columns={'index': 'Model'})
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

In [ ]:
sns.barplot(data=plot_df, x='Model', y='MAE', color='steelblue', ax=axes[0])
axes[0].axhline(baseline_mae, color='red', linestyle='--', label=f'Baseline ({baseline_mae:.3f})')
axes[0].set_title('Model Comparison MAE')
axes[0].tick_params(axis='x', rotation=35)
axes[0].legend()

In [ ]:
sns.barplot(data=plot_df, x='Model', y='Spearman rho', color='seagreen', ax=axes[1])
axes[1].axhline(baseline_r, color='red', linestyle='--', label=f'Baseline ({baseline_r:.3f})')
axes[1].set_title('Model Comparison Spearman rho')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend()

In [ ]:
plt.tight_layout()
plt.savefig(BASE / 'outputs/model_accuracy_comparison.png', dpi=300)
plt.show()

In [ ]:
# predicted vs actual for best GB
plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_gb_preds, alpha=0.3, s=15, color='steelblue')
plt.plot([1, 20], [1, 20], 'r--', linewidth=1.2, label='Perfect prediction')
plt.xlabel('Actual Finishing Position')
plt.ylabel('Predicted Finishing Position')
plt.title('Gradient Boosting Predicted vs Actual (2024-2025)')
plt.legend()
plt.tight_layout()
plt.savefig(BASE / 'outputs/predicted_vs_actual.png', dpi=300)
plt.show()

In [ ]:
# SHAP, tree explainer is exact for gradient boosting
explainer   = shap.TreeExplainer(best_gb_model)
shap_values = explainer.shap_values(X_test_scaled.values)

In [ ]:
shap_imp = pd.DataFrame({'feature': FEATURES,
                         'mean_abs_shap': np.abs(shap_values).mean(axis=0)
                         }).sort_values('mean_abs_shap', ascending=False)
print(shap_imp.to_string(index=False))

In [ ]:
shap.summary_plot(shap_values, X_test_scaled.values, feature_names=FEATURES, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(BASE / 'outputs/shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test_scaled.values, feature_names=FEATURES, show=False)
plt.tight_layout()
plt.savefig(BASE / 'outputs/shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# partial dependance plot
fig, ax = plt.subplots(figsize=(12, 5))
PartialDependenceDisplay.from_estimator(best_gb_model, X_test_scaled, FEATURES, ax=ax)
plt.suptitle('Partial Dependence Gradient Boosting (2024-2025)')
plt.tight_layout()
plt.savefig(BASE / 'outputs/partial_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print('Done.')